# K3D Spherical (Radius-Based) Clipping

Interactive controls for spherical clipping of 3D volume data.

In [1]:
import numpy as np
import k3d
from ipywidgets import interact, widgets
from IPython.display import display
import sys
sys.path.append('.')
from k3d_spherical_clipping import SphericalClippingController

In [2]:
# Initialize controller
controller = SphericalClippingController()
controller.plot.display()

def update_sphere(method, radius, center_x, center_y, center_z, clip_inside, 
                 octant_enabled, octant_x, octant_y, octant_z, 
                 log_scale, log_range, resolution):
    controller.method = method
    controller.sphere_radius = radius
    controller.sphere_center = [center_x, center_y, center_z]
    controller.clip_inside = clip_inside
    controller.polyhedron_resolution = resolution
    
    # Handle octant cutting (excludes selected octant)
    controller.octant_cut = octant_enabled
    if octant_enabled:
        controller.octant_mode = 'custom'
        controller.octant_signs = [
            1 if octant_x else -1,
            1 if octant_y else -1, 
            1 if octant_z else -1
        ]
    
    # Handle log scaling
    controller.log_scale = log_scale
    controller.log_dynamic_range = log_range
    
    if method == 'polyhedron':
        controller.generate_polyhedron_planes()
    controller.update_clipping()
    
interact(update_sphere,
    method=widgets.RadioButtons(
        options=['polyhedron', 'masking', 'hybrid'],
        value='masking',  # Changed default to masking (works better with octants)
        description='Method:'
    ),
    radius=widgets.FloatSlider(
        min=1, max=60, value=15, description='Radius:'
    ),
    center_x=widgets.FloatSlider(
        min=0, max=controller.h, value=controller.h/2, description='Center X:'
    ),
    center_y=widgets.FloatSlider(
        min=0, max=controller.k, value=controller.k/2, description='Center Y:'
    ),
    center_z=widgets.FloatSlider(
        min=0, max=controller.l, value=controller.l/2, description='Center Z:'
    ),
    clip_inside=widgets.Checkbox(
        value=True, description='Show Inside'
    ),
    octant_enabled=widgets.Checkbox(
        value=False, description='Exclude Octant'
    ),
    octant_x=widgets.Checkbox(
        value=True, description='X > center (to exclude)'
    ),
    octant_y=widgets.Checkbox(
        value=True, description='Y > center (to exclude)'
    ),
    octant_z=widgets.Checkbox(
        value=True, description='Z > center (to exclude)'
    ),
    log_scale=widgets.Checkbox(
        value=False, description='Log Scale Intensity'
    ),
    log_range=widgets.FloatSlider(
        min=1, max=1000, value=100, 
        description='Log Range:',
        tooltip='Dynamic range factor for log scaling'
    ),
    resolution=widgets.IntSlider(
        min=0, max=3, value=2, description='Resolution:'
    )
);

Loading data...
Data loaded: 81 × 81 × 81
Generated 20 planes for sphere approximation


Output()

interactive(children=(RadioButtons(description='Method:', index=1, options=('polyhedron', 'masking', 'hybrid')…

## Spherical Clipping Controls

In [3]:
# Animation controls
def animate():
    controller.animate_radius(min_radius=5, max_radius=25, steps=30, delay=0.03)
    
animate_btn = widgets.Button(description='Animate Radius')
animate_btn.on_click(lambda b: animate())
display(animate_btn)

Button(description='Animate Radius', style=ButtonStyle())

In [ ]:
# Camera animation controls
from ipywidgets import Button, HBox, VBox, FloatSlider, Dropdown, Label

# Animation buttons
orbit_btn = Button(description="Orbit", button_style='primary')
zoom_btn = Button(description="Zoom In", button_style='primary')
reveal_btn = Button(description="Zoom Reveal", button_style='success')
octant_btn = Button(description="Octant Tour", button_style='info')
anisotropy_btn = Button(description="Anisotropy", button_style='warning')
stop_btn = Button(description="Stop", button_style='danger')
reset_btn = Button(description="Reset Camera")

# Preset views dropdown
preset_dropdown = Dropdown(
    options=['front', 'back', 'left', 'right', 'top', 'bottom', 'isometric', 'optimal'],
    value='isometric',
    description='Preset View:'
)

# Animation parameters
duration_slider = FloatSlider(
    min=2.0, max=20.0, value=6.0, step=0.5,
    description='Duration (s):'
)

elevation_slider = FloatSlider(
    min=0, max=90, value=45, step=5,
    description='Elevation:'
)

radius_slider = FloatSlider(
    min=1.0, max=5.0, value=2.5, step=0.1,
    description='Distance:'
)

# Button callbacks
def on_orbit_click(b):
    controller.store_initial_camera()
    controller.add_orbital_camera(
        duration=duration_slider.value,
        radius_factor=radius_slider.value,
        elevation=elevation_slider.value
    )

def on_zoom_click(b):
    controller.store_initial_camera()
    controller.add_zoom_animation(
        duration=duration_slider.value,
        elevation=elevation_slider.value
    )

def on_reveal_click(b):
    controller.store_initial_camera()
    controller.create_zoom_reveal_animation(
        duration=duration_slider.value
    )

def on_octant_click(b):
    controller.store_initial_camera()
    controller.create_octant_inspection_animation(
        duration=duration_slider.value * 1.5
    )

def on_anisotropy_click(b):
    controller.store_initial_camera()
    controller.add_anisotropy_showcase_animation(
        duration=duration_slider.value * 1.5
    )

def on_stop_click(b):
    if controller.plot:
        controller.plot.stop_auto_play()
        controller.is_animating = False

def on_reset_click(b):
    controller.reset_camera()

def on_preset_change(change):
    controller.set_preset_view(change['new'], transition_duration=1.0)

# Connect callbacks
orbit_btn.on_click(on_orbit_click)
zoom_btn.on_click(on_zoom_click)
reveal_btn.on_click(on_reveal_click)
octant_btn.on_click(on_octant_click)
anisotropy_btn.on_click(on_anisotropy_click)
stop_btn.on_click(on_stop_click)
reset_btn.on_click(on_reset_click)
preset_dropdown.observe(on_preset_change, names='value')

# Layout
animation_buttons = HBox([orbit_btn, zoom_btn, reveal_btn, octant_btn, anisotropy_btn])
control_buttons = HBox([stop_btn, reset_btn])
parameters = VBox([duration_slider, elevation_slider, radius_slider])
presets = HBox([preset_dropdown])

# Display all controls
display(VBox([
    Label(value="Animation Controls:"),
    animation_buttons,
    control_buttons,
    Label(value="Parameters:"),
    parameters,
    Label(value="Preset Views:"),
    presets
]))

## Camera Animation Controls

Interactive controls for cinematic camera animations.